In [4]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hyper-parameters ─────────────────────────────────────────────────────────
MODEL_NAME  = "microsoft/codebert-base"
MAX_LENGTH  = 512
BATCH_SIZE  = 16     # lower to 8 for <8 GB VRAM
EPOCHS      = 5
LR          = 2e-5   # best for fine-tuning BERT-family models
WARMUP_FRAC = 0.1    # 10% warmup steps
DROPOUT     = 0.3
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print("\nHyper-parameters set ✓")

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu

Hyper-parameters set ✓


In [5]:
class CodeDefectDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data       = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        code  = str(self.data.loc[idx, 'clean_code'])
        label = int(self.data.loc[idx, 'target'])
        enc   = self.tokenizer(code, max_length=self.max_length,
                               padding='max_length', truncation=True,
                               return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(label, dtype=torch.long)
        }


tokenizer  = AutoTokenizer.from_pretrained('tokenizer/')
train_df   = pd.read_csv('data/train_clean.csv')
valid_df   = pd.read_csv('data/valid_clean.csv')

train_dataset = CodeDefectDataset(train_df, tokenizer, MAX_LENGTH)
valid_dataset = CodeDefectDataset(valid_df, tokenizer, MAX_LENGTH)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
valid_loader  = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

Train batches: 1366 | Valid batches: 171


In [6]:
class CodeReviewModel(nn.Module):
    """
    CodeBERT fine-tuned for binary defect classification.
    Architecture: CodeBERT → [CLS] → MLP head → 2 classes
    """

    def __init__(self, model_name: str, num_labels: int = 2, dropout: float = 0.3):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden_size    = self.encoder.config.hidden_size  # 768 for codebert-base

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        logits     = self.classifier(cls_output)
        return logits


model = CodeReviewModel(MODEL_NAME, dropout=DROPOUT).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6002.55it/s]


Total parameters:     124,843,010
Trainable parameters: 124,843,010


In [7]:
# Compute class weights to handle imbalance
labels_array = train_df['target'].values
class_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(labels_array),
    y            = labels_array
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Class weights: Clean={class_weights[0]:.3f} | Defective={class_weights[1]:.3f}")

# Loss, Optimizer, Scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Total training steps: {total_steps} | Warmup steps: {warmup_steps}")

Class weights: Clean=0.923 | Defective=1.091
Total training steps: 6830 | Warmup steps: 683


In [8]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(loader, desc='Training', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch in tqdm(loader, desc='Validating', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits      = model(input_ids, attention_mask)
            loss        = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total


print("Training functions defined ✓")

Training functions defined ✓
